In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
orders               = spark.table(f"{catalogo}.{esquema_source}.orders")
order_products_train = spark.table(f"{catalogo}.{esquema_source}.order_products__train")
order_products_prior = spark.table(f"{catalogo}.{esquema_source}.order_products__prior")
products             = spark.table(f"{catalogo}.{esquema_source}.products")
aisles               = spark.table(f"{catalogo}.{esquema_source}.aisles")
departments          = spark.table(f"{catalogo}.{esquema_source}.departments")

## Ingesta de tablas Silver

In [0]:
# Silver Orders
silver_orders = (
    bronze_orders
    .select(
        "order_id", 
        "user_id", 
        "order_number", 
        "order_dow",
        "order_hour_of_day", 
        "days_since_prior_order"
    )
    .withColumn(
        "order_day_name",
        F.when(F.col("order_dow") == 0, "Sunday")
         .when(F.col("order_dow") == 1, "Monday")
         .when(F.col("order_dow") == 2, "Tuesday")
         .when(F.col("order_dow") == 3, "Wednesday")
         .when(F.col("order_dow") == 4, "Thursday")
         .when(F.col("order_dow") == 5, "Friday")
         .otherwise("Saturday")
    )
    .withColumn(
        "order_time_category",
        F.when(F.col("order_hour_of_day").between(6, 11), "Morning")
         .when(F.col("order_hour_of_day").between(12, 17), "Afternoon")
         .when(F.col("order_hour_of_day").between(18, 23), "Evening")
         .otherwise("Night")
    )
)
silver_orders.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.silver_orders")



In [0]:
# Silver Order Products
silver_order_products = (
    bronze_order_products__train
        .select(
                "order_id", 
                "product_id", 
                "add_to_cart_order", 
                "reordered"
                )
                .withColumn("order_type", F.lit("train"))
                
    .union(bronze_order_products__prior
           .select(
                    "order_id", 
                    "product_id", 
                    "add_to_cart_order", 
                    "reordered"
                    )
                    .withColumn("order_type", F.lit("prior"))
           )
    .withColumn(
        "cart_position_category",
        F.when(F.col("add_to_cart_order") == 1, "First in cart")
         .when(F.col("add_to_cart_order") <= 3, "Early in cart")
         .otherwise("Later in cart")
    )
    .withColumn(
        "reorder_flag",
        F.when(F.col("reordered") == 1, "Reordered").otherwise("First time")
    )
)
silver_order_products.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.silver_order_products")



In [0]:
# Silver Products
silver_products = (
    bronze_products
    .join(bronze_aisle, "aisle_id", "left")
    .join(bronze_department, "department_id", "left")
    .select(
        "product_id", 
        "product_name", 
        "aisle_id", 
        "aisle",
        "department_id", 
        "department"
    )
    .withColumn(
        "product_category_group",
        F.when(F.col("department").isin("produce", "dairy eggs", "meat seafood"), "Fresh")
         .when(F.col("department").isin("snacks", "beverages", "pantry"), "Packaged")
         .otherwise("Other")
    )
)
silver_products.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.silver_products")

In [0]:
# Silver Orders Products Detail
silver_orders_products_detail = (
    silver_orders
    .join(silver_order_products, "order_id", "inner")
    .join(silver_products, "product_id", "inner")
)
silver_orders_products_detail.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.silver_orders_products_detail")